In [1]:
import os
import time
from typing import Dict, Tuple, List

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [2]:
# ============================================================
#  Dataset
# ============================================================

class DocumentShadowDataset(Dataset):
    """
    Dataset untuk shadow removal dokumen.

    CSV diharapkan punya kolom:
      - 'img' : path/filename shadow image relatif terhadap img_root
      - 'gt'  : path/filename ground truth (non-shadow) relatif terhadap img_root

    Kalau nama kolom di CSV beda, tinggal sesuaikan di __getitem__.
    """

    def __init__(
        self,
        csv_path: str,
        img_root: str,
        img_size: int = 512,
    ):
        super().__init__()
        self.df = pd.read_csv(csv_path)
        if "img" not in self.df.columns or "gt" not in self.df.columns:
            raise ValueError("CSV must contain 'img' and 'gt' columns.")

        self.img_root = img_root
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _load_rgb(self, rel_path: str) -> np.ndarray:
        # jika path sudah absolute / mengandung folder dataset, pakai langsung
        if rel_path.startswith("./") or rel_path.startswith("/") or "dataset" in rel_path:
            full_path = rel_path  # jangan join dengan img_root
        else:
            full_path = os.path.join(self.img_root, rel_path)
    
        img = cv2.imread(full_path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Image not found: {full_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        return img

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        shadow_rel = row["img"]
        gt_rel = row["gt"]

        shadow = self._load_rgb(shadow_rel)
        gt = self._load_rgb(gt_rel)

        # to tensor, range [0,1]
        shadow_t = torch.from_numpy(shadow).permute(2, 0, 1).float() / 255.0
        gt_t = torch.from_numpy(gt).permute(2, 0, 1).float() / 255.0

        # normalize ke [-1,1] biar konsisten dengan tanh di output
        shadow_t = shadow_t * 2.0 - 1.0
        gt_t = gt_t * 2.0 - 1.0

        base_name = os.path.splitext(os.path.basename(shadow_rel))[0]

        return shadow_t, gt_t, base_name


In [3]:
# ============================================================
#  Model blocks: BENet, SRNet, BEDSRNet
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, norm: bool = True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.conv(x)
        p = self.pool(x)
        return x, p


class UpBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.size(-1) != skip.size(-1) or x.size(-2) != skip.size(-2):
            diff_y = skip.size(-2) - x.size(-2)
            diff_x = skip.size(-1) - x.size(-1)
            x = F.pad(
                x,
                [diff_x // 2, diff_x - diff_x // 2,
                 diff_y // 2, diff_y - diff_y // 2],
            )
        x = torch.cat([skip, x], dim=1)
        x = self.conv(x)
        return x


class BENet(nn.Module):
    """
    Brightness & attention estimator:
      - Input : shadow image [B,3,H,W] (range [-1,1])
      - Output:
          bg   : [B,3] (background color, di [-1,1])
          attn : [B,1,h,w] (attention map, sigmoid [0,1])
    """

    def __init__(self, in_ch: int = 3, base_ch: int = 32):
        super().__init__()
        self.enc1 = DownBlock(in_ch, base_ch)
        self.enc2 = DownBlock(base_ch, base_ch * 2)
        self.enc3 = DownBlock(base_ch * 2, base_ch * 4)

        self.bottleneck = ConvBlock(base_ch * 4, base_ch * 8)

        self.attention_head = nn.Conv2d(base_ch * 8, 1, kernel_size=1)
        self.bg_fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(base_ch * 8, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        _, p1 = self.enc1(x)
        _, p2 = self.enc2(p1)
        _, p3 = self.enc3(p2)
        b = self.bottleneck(p3)

        attn = self.attention_head(b)       # [B,1,h,w]
        attn = torch.sigmoid(attn)

        bg = self.bg_fc(b)                  # [B,3,1,1]
        bg = bg.view(bg.size(0), 3)         # [B,3]

        return bg, attn


class SRNet(nn.Module):
    """
    Shadow removal U-Net:
      - Input : concat(shadow, attention_map_up, bg_map) -> [B,7,H,W]
      - Output: non-shadow image [B,3,H,W] in [-1,1] (after tanh)
    """

    def __init__(self, in_ch: int = 7, base_ch: int = 64):
        super().__init__()
        self.down1 = DownBlock(in_ch, base_ch)
        self.down2 = DownBlock(base_ch, base_ch * 2)
        self.down3 = DownBlock(base_ch * 2, base_ch * 4)
        self.down4 = DownBlock(base_ch * 4, base_ch * 8)

        self.bottleneck = ConvBlock(base_ch * 8, base_ch * 16)

        self.up4 = UpBlock(base_ch * 16, base_ch * 8)
        self.up3 = UpBlock(base_ch * 8, base_ch * 4)
        self.up2 = UpBlock(base_ch * 4, base_ch * 2)
        self.up1 = UpBlock(base_ch * 2, base_ch)

        self.out_conv = nn.Conv2d(base_ch, 3, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1, p1 = self.down1(x)
        x2, p2 = self.down2(p1)
        x3, p3 = self.down3(p2)
        x4, p4 = self.down4(p3)

        b = self.bottleneck(p4)

        u4 = self.up4(b, x4)
        u3 = self.up3(u4, x3)
        u2 = self.up2(u3, x2)
        u1 = self.up1(u2, x1)

        out = self.out_conv(u1)
        out = torch.tanh(out)  # output range [-1,1]
        return out


class BEDSRNet(nn.Module):
    """
    Wrapper:
      - shadow -> BENet -> (bg, attn)
      - concat shadow + attn_up + bg_map -> SRNet -> nonshadow
      - forward() return (nonshadow, bg, attn_up)
    """

    def __init__(self):
        super().__init__()
        self.benet = BENet()
        self.srnet = SRNet()

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        bg, attn = self.benet(x)  # bg [B,3], attn [B,1,h,w]

        attn_up = F.interpolate(
            attn,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        bg_map = bg.view(bg.size(0), 3, 1, 1)
        bg_map = bg_map.expand(-1, -1, x.size(2), x.size(3))

        gen_input = torch.cat([x, attn_up, bg_map], dim=1)  # [B,7,H,W]

        nonshadow = self.srnet(gen_input)

        return nonshadow, bg, attn_up



In [4]:
# ============================================================
#  Metrics: PSNR & SSIM (untuk range [-1,1])
# ============================================================

def _to_01(t: torch.Tensor) -> torch.Tensor:
    # [-1,1] -> [0,1]
    return (t + 1.0) / 2.0


def calc_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    pred_01 = _to_01(pred)
    tgt_01 = _to_01(target)
    mse = torch.mean((pred_01 - tgt_01) ** 2)
    if mse.item() == 0:
        return 100.0
    psnr = 20 * torch.log10(torch.tensor(1.0, device=pred.device)) - 10 * torch.log10(mse)
    return psnr.item()


def calc_ssim(pred: torch.Tensor, target: torch.Tensor) -> float:
    from torch.nn.functional import conv2d

    pred_01 = _to_01(pred)
    tgt_01 = _to_01(target)

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    weights = torch.tensor([0.299, 0.587, 0.114],
                           device=pred.device).view(1, 3, 1, 1)
    pred_gray = (pred_01 * weights).sum(dim=1, keepdim=True)
    tgt_gray = (tgt_01 * weights).sum(dim=1, keepdim=True)

    kernel = torch.ones((1, 1, 11, 11), device=pred.device) / (11 * 11)

    mu1 = conv2d(pred_gray, kernel, padding=5)
    mu2 = conv2d(tgt_gray, kernel, padding=5)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = conv2d(pred_gray * pred_gray, kernel, padding=5) - mu1_sq
    sigma2_sq = conv2d(tgt_gray * tgt_gray, kernel, padding=5) - mu2_sq
    sigma12 = conv2d(pred_gray * tgt_gray, kernel, padding=5) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / (
        (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2) + 1e-8
    )
    return ssim_map.mean().item()


In [5]:
# ============================================================
#  Logging ke TXT + Plot
# ============================================================

def init_log_file(log_path: str):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w") as f:
        f.write(
            "epoch\t"
            "train_loss\t"
            "val_loss\t"
            "val_psnr\t"
            "val_ssim\n"
        )


def append_log_epoch(
    log_path: str,
    epoch: int,
    train_loss: float,
    val_loss: float,
    val_psnr: float,
    val_ssim: float,
):
    with open(log_path, "a") as f:
        f.write(
            f"{epoch}\t"
            f"{train_loss:.6f}\t"
            f"{val_loss:.6f}\t"
            f"{val_psnr:.4f}\t"
            f"{val_ssim:.4f}\n"
        )


def save_training_plots(history: Dict[str, List[float]], run_id: str, root_dir: str = "resultplot"):
    out_dir = os.path.join(root_dir, f"training{run_id}")
    os.makedirs(out_dir, exist_ok=True)

    epochs = history["epoch"]
    train_loss = history["train_loss"]
    val_loss = history["val_loss"]
    val_psnr = history["val_psnr"]
    val_ssim = history["val_ssim"]

    # Plot 1: train & val loss
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, train_loss, label="Train Loss")
    plt.plot(epochs, val_loss, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("L1 Loss")
    plt.title(f"Training vs Validation Loss (training{run_id})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "loss_curve.png"))
    plt.close()

    # Plot 2: PSNR & SSIM (val)
    plt.figure(figsize=(6, 4))
    plt.plot(epochs, val_psnr, label="Val PSNR (dB)")
    plt.plot(epochs, val_ssim, label="Val SSIM")
    plt.xlabel("Epoch")
    plt.ylabel("Metric value")
    plt.title(f"PSNR & SSIM (validation) – training{run_id}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "metrics_curve.png"))
    plt.close()


In [6]:
# ============================================================
#  Train & Eval
# ============================================================

scaler = GradScaler()

def train_one_epoch(model, loader, optimizer, device, epoch, num_epochs):
    model.train()
    total_loss = 0.0
    total_samples = 0
    total_batches = len(loader)

    print(f"\n===== Epoch {epoch} / {num_epochs} =====")
    print(f"Total batches: {total_batches}")

    for batch_idx, (shadow, gt, _) in enumerate(loader, start=1):
        shadow = shadow.to(device)
        gt = gt.to(device)

        optimizer.zero_grad(set_to_none=True)

        # autocast versi kompatibel (tanpa device_type)
        with autocast(enabled=(device.type == "cuda")):
            pred, bg, att = model(shadow)
            loss = F.l1_loss(pred, gt)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bsz = shadow.size(0)
        total_loss += loss.item() * bsz
        total_samples += bsz

        progress = batch_idx / total_batches
        bar_len = 30
        filled = int(progress * bar_len)
        bar = "=" * filled + "-" * (bar_len - filled)
        print(
            f"\r[Epoch {epoch}] [{bar}] "
            f"{batch_idx}/{total_batches} "
            f"loss={loss.item():.4f}",
            end="",
        )

    print("")
    return total_loss / max(1, total_samples)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_ssim = 0.0
    total_samples = 0

    for shadow, gt, _ in loader:
        shadow = shadow.to(device)
        gt = gt.to(device)

        pred, bg, att = model(shadow)
        loss = F.l1_loss(pred, gt)

        psnr = calc_psnr(pred, gt)
        ssim = calc_ssim(pred, gt)

        bsz = shadow.size(0)
        total_loss += loss.item() * bsz
        total_psnr += psnr * bsz
        total_ssim += ssim * bsz
        total_samples += bsz

    if total_samples == 0:
        return 0.0, 0.0, 0.0

    return (
        total_loss / total_samples,
        total_psnr / total_samples,
        total_ssim / total_samples,
    )


C:\Users\Lenovo Legion\AppData\Local\Temp\ipykernel_7524\173476683.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [7]:
# ============================================================
#  Inference & save 4 outputs
# ============================================================

@torch.no_grad()
def run_test_and_save_four_outputs(
    model,
    loader,
    device,
    save_root: str,
):
    """
    Simpan 4 output per gambar:
      - *_shadow.png
      - *_nonshadow.png
      - *_attention.png
      - *_background.png
    Ke folder: processoutput/trainingX/
    """
    os.makedirs(save_root, exist_ok=True)
    model.eval()

    for shadow, gt, base_name in loader:
        shadow = shadow.to(device)
        pred, bg, att = model(shadow)

        shadow_01 = _to_01(shadow)
        pred_01 = _to_01(pred)

        for i in range(shadow.size(0)):
            name = base_name[i]

            # shadow
            shadow_img = shadow_01[i].cpu().permute(1, 2, 0).numpy()
            shadow_img = (shadow_img * 255.0).clip(0, 255).astype(np.uint8)
            shadow_bgr = cv2.cvtColor(shadow_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(
                os.path.join(save_root, f"{name}_shadow.png"),
                shadow_bgr,
            )

            # non-shadow
            pred_img = pred_01[i].cpu().permute(1, 2, 0).numpy()
            pred_img = (pred_img * 255.0).clip(0, 255).astype(np.uint8)
            pred_bgr = cv2.cvtColor(pred_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(
                os.path.join(save_root, f"{name}_nonshadow.png"),
                pred_bgr,
            )

            # attention map
            att_map = att[i, 0].cpu().numpy()
            att_norm = (att_map * 255.0).clip(0, 255).astype(np.uint8)
            att_color = cv2.applyColorMap(att_norm, cv2.COLORMAP_JET)
            cv2.imwrite(
                os.path.join(save_root, f"{name}_attention.png"),
                att_color,
            )

            # background color map (bg: [-1,1] -> [0,1])
            bg_color = _to_01(bg[i].cpu()).numpy()
            H, W = att_map.shape
            bg_map = np.ones((H, W, 3), dtype=np.float32)
            bg_map[..., 0] *= bg_color[0]
            bg_map[..., 1] *= bg_color[1]
            bg_map[..., 2] *= bg_color[2]
            bg_img = (bg_map * 255.0).clip(0, 255).astype(np.uint8)
            bg_bgr = cv2.cvtColor(bg_img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(
                os.path.join(save_root, f"{name}_background.png"),
                bg_bgr,
            )

In [8]:
# ============================================================
#  Runner: trainingX (log, plots, 4 outputs)
# ============================================================

def run_training_with_logging(
    run_id: str = "1",
    train_csv: str = "./csv/Jung/train.csv",
    val_csv: str = "./csv/Jung/val.csv",
    test_csv: str = "./csv/Jung/test.csv",
    img_root: str = "./dataset/Jung",
    img_size: int = 512,
    batch_size: int = 4,
    num_workers: int = 0,
    num_epochs: int = 50,
    lr: float = 1e-4,
):
    """
    run_id -> menentukan:
      - trainlog/training{run_id}.txt
      - resultplot/training{run_id}/loss_curve.png & metrics_curve.png
      - processoutput/training{run_id}/*.png
      - checkpoints/bedsrnet_jung_best_{run_id}.pth
      - best_model/training{run_id}/best_model.pth
      - best_model/training{run_id}/last_model.pth
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # ===== Dataset & DataLoader =====
    train_ds = DocumentShadowDataset(train_csv, img_root, img_size)
    val_ds   = DocumentShadowDataset(val_csv,   img_root, img_size)
    test_ds  = DocumentShadowDataset(test_csv,  img_root, img_size)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    model = BEDSRNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ===== Logging TXT =====
    os.makedirs("trainlog", exist_ok=True)
    log_path = os.path.join("trainlog", f"training{run_id}.txt")
    init_log_file(log_path)

    # ===== History untuk plot =====
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_psnr": [],
        "val_ssim": [],
    }

    # ===== Save best model (checkpoints + best_model folder) =====
    best_score = 0.0
    os.makedirs("checkpoints", exist_ok=True)
    best_ckpt_path = os.path.join("checkpoints", f"bedsrnet_jung_best_{run_id}.pth")

    # folder best_model/trainingX
    best_model_dir = os.path.join("best_model", f"training{run_id}")
    os.makedirs(best_model_dir, exist_ok=True)
    best_model_path = os.path.join(best_model_dir, "best_model.pth")
    last_model_path = os.path.join(best_model_dir, "last_model.pth")

    # ===== Training loop =====
    for epoch in range(1, num_epochs + 1):
        start_time = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch, num_epochs)
        val_loss, val_psnr, val_ssim = evaluate(model, val_loader, device)
        elapsed = time.time() - start_time

        print(
            f"[Run {run_id}] [Epoch {epoch:03d}/{num_epochs}] "
            f"time={elapsed:.1f}s | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_psnr={val_psnr:.2f} dB | "
            f"val_ssim={val_ssim:.4f}"
        )

        # update history
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_psnr"].append(val_psnr)
        history["val_ssim"].append(val_ssim)

        # tulis ke txt
        append_log_epoch(
            log_path,
            epoch,
            train_loss,
            val_loss,
            val_psnr,
            val_ssim,
        )

        # skor gabungan untuk pilih best model
        score = val_psnr + (val_ssim * 10.0)
        if score > best_score:
            best_score = score

            # simpan ke checkpoints lama (biar kalau perlu compat ke script lain)
            torch.save(model.state_dict(), best_ckpt_path)

            # simpan juga ke best_model/trainingX/best_model.pth
            torch.save(model.state_dict(), best_model_path)

            print(
                f"  -> Save BEST model:"
                f"\n       {best_ckpt_path}"
                f"\n       {best_model_path}"
                f"\n     (PSNR={val_psnr:.2f}, SSIM={val_ssim:.4f})"
            )

    # ===== Save LAST model di best_model/trainingX/last_model.pth =====
    torch.save(model.state_dict(), last_model_path)
    print(f"  -> Save LAST model: {last_model_path}")

    # ===== Simpan plot training =====
    save_training_plots(history, run_id)

    # ===== Inference & simpan 4 output =====
    save_root = os.path.join("processoutput", f"training{run_id}")
    print(f"Run inference test set & save outputs to: {save_root}")
    run_test_and_save_four_outputs(model, test_loader, device, save_root)

    print(f"Done.")
    print(f"  Log      : {log_path}")
    print(f"  Plots    : resultplot/training{run_id}/")
    print(f"  Best     : {best_model_path}")
    print(f"  Last     : {last_model_path}")
    print(f"  Outputs  : processoutput/training{run_id}/")


In [9]:
# ============================================================
#  Main 
# ============================================================

if __name__ == "__main__":
    run_training_with_logging(
        run_id="3",
        train_csv="./csv/Jung/train.csv",
        val_csv="./csv/Jung/val.csv",
        test_csv="./csv/Jung/test.csv",
        img_root="./dataset/Jung",
        img_size=512,
        batch_size=4,
        num_workers=0,
        num_epochs=100,
        lr=1e-4,
    )

Device: cuda

===== Epoch 1 / 100 =====
Total batches: 15


C:\Users\Lenovo Legion\AppData\Local\Temp\ipykernel_7524\173476683.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):


[Epoch 1] [==============================] 15/15 loss=0.4966
[Run 3] [Epoch 001/100] time=5.5s | train_loss=0.4835 | val_loss=0.5393 | val_psnr=10.94 dB | val_ssim=0.5664
  -> Save BEST model:
       checkpoints\bedsrnet_jung_best_3.pth
       best_model\training3\best_model.pth
     (PSNR=10.94, SSIM=0.5664)

===== Epoch 2 / 100 =====
Total batches: 15
[Epoch 2] [==============================] 15/15 loss=0.3110
[Run 3] [Epoch 002/100] time=4.8s | train_loss=0.3300 | val_loss=0.2758 | val_psnr=16.23 dB | val_ssim=0.7026
  -> Save BEST model:
       checkpoints\bedsrnet_jung_best_3.pth
       best_model\training3\best_model.pth
     (PSNR=16.23, SSIM=0.7026)

===== Epoch 3 / 100 =====
Total batches: 15
[Epoch 3] [==============================] 15/15 loss=0.2166
[Run 3] [Epoch 003/100] time=4.8s | train_loss=0.2526 | val_loss=0.1975 | val_psnr=17.79 dB | val_ssim=0.8578
  -> Save BEST model:
       checkpoints\bedsrnet_jung_best_3.pth
       best_model\training3\best_model.pth
     (PS